# Pose Extraction using YOLO (RWF-2000)

In this notebook, we extract human pose keypoints from video data using YOLO pose estimation.

## What is done

- Load cleaned RGB dataset (videos)
- Run YOLO pose model on each frame
- Detect people and extract 17 keypoints per person
- Select top 2 persons per frame
- Save results as `.npy` files

## Output

Saved pose data:
data/processed/pose/npy/

Each file shape:
(T, 2, 17, 3)

Where:
- T = number of frames
- 2 = max number of persons
- 17 = keypoints
- 3 = (x, y, confidence)

Setup + Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/violence-detection"

# Copy dataset from Drive to Colab local storage

!cp -r /content/drive/MyDrive/violence-detection/data/rwf2000_clean /content/

In [ ]:
DATA_ROOT = "/content/rwf2000_clean"   # RGB videos (copied to Colab)
OUT_ROOT = "/content/rwf2000_yolo_pose"

SPLITS = ["train", "val"]
CLASSES = ["Fight", "NonFight"]

os.makedirs(OUT_ROOT, exist_ok=True)

for split in SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUT_ROOT, split, cls), exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT :", OUT_ROOT)

Install + import YOLO

In [ ]:
!pip -q install ultralytics opencv-python-headless tqdm

In [ ]:
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from ultralytics import YOLO

Load YOLO pose model

In [ ]:
MODEL_NAME = "yolo11n-pose.pt"
model = YOLO(MODEL_NAME)

print("Loaded:", MODEL_NAME)

Person selection (keep top 2)

In [ ]:
def choose_top2_persons(boxes_xyxy, keypoints_conf, max_persons=2):
    """
    Return indices of top persons based on:
    bbox area * average keypoint confidence
    """
    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        return []

    scores = []

    for i in range(len(boxes_xyxy)):
        x1, y1, x2, y2 = boxes_xyxy[i]
        area = max(0.0, x2 - x1) * max(0.0, y2 - y1)

        if keypoints_conf is not None and len(keypoints_conf) > i:
            conf = float(np.mean(keypoints_conf[i]))
        else:
            conf = 1.0

        score = area * conf
        scores.append((score, i))

    scores.sort(reverse=True)  # highest first
    top_indices = [idx for _, idx in scores[:max_persons]]
    return top_indices

Extract keypoints from video

In [ ]:
def extract_video_keypoints(video_path, model, conf=0.25, imgsz=640, max_frames=None, frame_stride=1):
    """
    Returns:
        keypoints: np.ndarray of shape (T, 17, 3)  where 3 = (x, y, confidence)
        fps: float
        total_frames_read: int
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = 0
    saved = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % frame_stride != 0:
            frame_idx += 1
            continue

        # Run YOLO pose on current frame
        results = model.predict(
            source=frame,
            conf=conf,
            imgsz=imgsz,
            verbose=False
        )

        # Default output if no person found
        frame_kpts = np.zeros((2, 17, 3), dtype=np.float32)

        if len(results) > 0:
            r = results[0]

            has_boxes = hasattr(r, "boxes") and r.boxes is not None and len(r.boxes) > 0
            has_kpts = hasattr(r, "keypoints") and r.keypoints is not None

            if has_boxes and has_kpts:
                boxes_xyxy = r.boxes.xyxy.cpu().numpy()  # (N, 4)

                # keypoints.xy -> (N, 17, 2)
                keypoints_xy = r.keypoints.xy.cpu().numpy()

                # keypoints.conf may be None in some situations
                keypoints_conf = None
                if getattr(r.keypoints, "conf", None) is not None:
                    keypoints_conf = r.keypoints.conf.cpu().numpy()  # (N, 17)

                top_idxs = choose_top2_persons(boxes_xyxy, keypoints_conf, max_persons=2)

                for person_slot, idx in enumerate(top_idxs):
                    frame_kpts[person_slot, :, :2] = keypoints_xy[idx][:17]

                    if keypoints_conf is not None:
                        frame_kpts[person_slot, :, 2] = keypoints_conf[idx][:17]
                    else:
                        frame_kpts[person_slot, :, 2] = 1.0

        saved.append(frame_kpts)

        frame_idx += 1
        if max_frames is not None and len(saved) >= max_frames:
            break

    cap.release()

    if len(saved) == 0:
        keypoints = np.zeros((0, 2, 17, 3), dtype=np.float32)
    else:
        keypoints = np.stack(saved, axis=0).astype(np.float32)

    return keypoints, fps, frame_idx

Process one split/class

In [ ]:
def process_split_class(split, cls, model, conf=0.25, imgsz=640, frame_stride=1, overwrite=False):
    in_dir = os.path.join(DATA_ROOT, split, cls)
    out_dir = os.path.join(OUT_ROOT, split, cls)

    video_paths = sorted(
        glob(os.path.join(in_dir, "*.avi")) +
        glob(os.path.join(in_dir, "*.mp4"))
    )

    print(f"{split}/{cls}: {len(video_paths)} videos")
    failed = []

    for vp in tqdm(video_paths, desc=f"{split}-{cls}"):
        base = os.path.splitext(os.path.basename(vp))[0]
        out_path = os.path.join(out_dir, base + ".npy")

        if os.path.exists(out_path) and not overwrite:
            continue

        try:
            kpts, fps, nread = extract_video_keypoints(
                video_path=vp,
                model=model,
                conf=conf,
                imgsz=imgsz,
                max_frames=None,
                frame_stride=frame_stride
            )
            np.save(out_path, kpts)

        except Exception as e:
            failed.append((vp, str(e)))

    return failed

Run extraction on dataset

In [ ]:
all_failed = []

for split in SPLITS:
    for cls in CLASSES:
        failed = process_split_class(
            split=split,
            cls=cls,
            model=model,
            conf=0.25,
            imgsz=640,
            frame_stride=1,
            overwrite=False
        )
        all_failed.extend(failed)

print("Done.")
print("Failed videos:", len(all_failed))

Save to Drive

In [ ]:
!cp -r /content/rwf2000_yolo_pose /content/drive/MyDrive/violence-detection/data/processed/pose/